# PPE Tracker — Test su Video

Questo notebook testa la pipeline completa:
1. **Setup** — verifica YOLO e VLM locale (SmolVLM via transformers)
2. **Frame sampling** — YOLO su frame campionati dal video
3. **Tracking completo** — pipeline con sliding window + escalation VLM
4. **Analisi alert** — visualizzazione dei frame con violazioni confermate

In [ ]:
import sys

sys.path.insert(0, "src")

from pathlib import Path
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from visual_security.analyzer import YOLOAnalyzer
from visual_security.vlm_validator import LocalVLMValidator
from visual_security.person_ppe_checker import PersonPPEChecker
from visual_security.video_tracker import build_tracker

# ── Config ────────────────────────────────────────────────────────────────────
VIDEO = r"C:\Users\a.damicis\Documents\projects\visual_security\ppe_video.mp4"
YOLO_MODEL = "weights/best.onnx"
# VLM locale in-process (transformers, nessun server).
VLM_MODEL = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"  # piu' accurato, piu' lento su CPU, ~9GB
# VLM_MODEL = "HuggingFaceTB/SmolVLM-500M-Instruct"  # ~2.5s/query su CPU
OUTPUT_VIDEO = "output/test_tracker_annotated.mp4"
ALERT_LOG = "output/test_tracker_alerts.json"

# ── Checks ────────────────────────────────────────────────────────────────────
assert Path(VIDEO).exists(), f"Video non trovato: {VIDEO}"
assert Path(YOLO_MODEL).exists(), f"Modello YOLO non trovato: {YOLO_MODEL}"

cap = cv.VideoCapture(VIDEO)
n_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv.CAP_PROP_FPS) or 25.0
fw = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
fh = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
cap.release()
duration = n_frames / fps

print(f"OK Video: {Path(VIDEO).name}")
print(f"   Risoluzione: {fw}x{fh}  |  {n_frames} frame  |  {fps:.1f} fps  |  {duration:.1f}s")
print(f"OK YOLO model: {YOLO_MODEL}")

if LocalVLMValidator.is_available():
    print(f"OK VLM locale: '{VLM_MODEL}' (i pesi vengono scaricati al primo utilizzo)")
else:
    print("WARN torch/transformers non installati -> VLM disabilitato")
    print("   pip install torch transformers pillow")

OK Video: ppe_video.mp4
   Risoluzione: 1280x720  |  240 frame  |  24.0 fps  |  10.0s
OK YOLO model: weights/dataset_1/yolo_nano_640/best.onnx


c:\Users\a.damicis\Documents\projects\visual_security\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK VLM locale: 'HuggingFaceTB/SmolVLM2-2.2B-Instruct' (i pesi vengono scaricati al primo utilizzo)


## 1 · Frame sampling — YOLO raw detection

Campiona N frame distribuiti nel video e mostra le detection YOLO per verificare che il modello riconosca correttamente persone e DPI.

In [ ]:
SAMPLES = 6  # numero di frame da campionare

yolo = YOLOAnalyzer(model_path=YOLO_MODEL, conf_threshold=0.35)
checker = PersonPPEChecker()

cap = cv.VideoCapture(VIDEO)
step = max(1, n_frames // SAMPLES)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i in range(SAMPLES):
    fi = i * step
    cap.set(cv.CAP_PROP_POS_FRAMES, fi)
    ret, frame = cap.read()
    if not ret:
        continue

    result = yolo.analyze(frame)
    person_results = checker.check(result.detections, fw, fh)

    vis = frame.copy()
    for det in result.detections:
        from visual_security.person_ppe_checker import _to_xyxy

        box = _to_xyxy(det.bbox, fw, fh)
        if box:
            x1, y1, x2, y2 = (int(v) for v in box)
            color = (0, 45, 210) if det.is_violation else (40, 200, 60)
            cv.rectangle(vis, (x1, y1), (x2, y2), color, 2)
            cv.putText(
                vis, f"{det.label} {det.confidence:.2f}", (x1, max(y1 - 6, 12)), cv.FONT_HERSHEY_SIMPLEX, 0.45, color, 1
            )

    n_persons = len(person_results)
    n_violation = sum(1 for p in person_results if not p.is_compliant)
    ts = fi / fps

    axes[i].imshow(cv.cvtColor(vis, cv.COLOR_BGR2RGB))
    axes[i].set_title(
        f"t={ts:.1f}s (frame {fi}) | {len(result.detections)} det | {n_persons} persone | {n_violation} violazioni",
        fontsize=9,
    )
    axes[i].axis("off")

    print(f"[t={ts:.1f}s] {len(result.detections)} det  {result.inference_time_ms:.0f}ms")
    for pr in person_results:
        print(f"  {pr.summary()}")

cap.release()
plt.tight_layout()
plt.show()

## 2 · Tracking completo

Lancia la pipeline completa:
- YOLO ogni N frame (sliding window)
- **PersonTracker**: identità persistente per persona (matching IoU tra frame, `track_id` stabile mostrato come `[T0]`, `[T1]`...) + **memoria PPE**: un DPI visto negli ultimi `ppe_memory_frames` frame è considerato ancora presente anche se YOLO lo perde temporaneamente (occlusioni/blur su guanti e scarpe)
- violazioni confermate solo se persistono per ≥ `persistence_frames` su `window_frames` (chiave = identità persona, non più cella di griglia)
- escalation al VLM locale (in-process) solo sulle violazioni confermate; la validazione gira su un thread in background, quindi non blocca l'elaborazione dei frame

Il video annotato viene salvato in `output/`.

In [ ]:
Path("output").mkdir(exist_ok=True)

vlm_model_arg = VLM_MODEL if LocalVLMValidator.is_available() else "none"

tracker = build_tracker(
    yolo_model_path=YOLO_MODEL,
    vlm_model=vlm_model_arg,
    persistence_frames=5,  # frame positivi necessari nella window
    window_frames=10,  # ampiezza sliding window
    skip_frames=3,  # YOLO ogni 3 frame (bilancia velocità/accuratezza)
    ppe_memory_frames=50,  # memoria PPE: un DPI visto negli ultimi ~2s e' considerato ancora presente
    display=False,  # no GUI nel notebook
    save_output=OUTPUT_VIDEO,
    alert_log=ALERT_LOG,
    yolo_conf=0.4,
    verbose=False,
)

print(f"Avvio tracking su {Path(VIDEO).name} ...")
alerts = tracker.run(VIDEO)

print(f"\n{'=' * 60}")
print(f"Tracking completato - {len(alerts)} alert confermati")
print(f"Video annotato -> {OUTPUT_VIDEO}")
print(f"Log JSON       -> {ALERT_LOG}")
print("=" * 60)
for a in alerts:
    print(a.summary())

## 3 · Analisi degli alert

Visualizza il frame corrispondente a ogni alert confermato con le violazioni evidenziate.

In [ ]:
if not alerts:
    print("Nessun alert - prova ad abbassare yolo_conf o persistence_frames.")
else:
    cap = cv.VideoCapture(VIDEO)
    cols = min(3, len(alerts))
    rows = (len(alerts) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    axes = np.array(axes).flatten()

    for i, alert in enumerate(alerts):
        cap.set(cv.CAP_PROP_POS_FRAMES, alert.frame_idx - 1)
        ret, frame = cap.read()
        if not ret:
            continue

        # Disegna bbox delle violazioni, colorato secondo il verdetto VLM
        # PER PERSONA (non il tag aggregato sull'intero alert):
        #   rosso   = confermata dal VLM (o VLM non ancora risolto/non configurato)
        #   viola   = il VLM ha scagionato questa persona (probabile falso positivo YOLO)
        any_refuted = False
        for pr in alert.violations:
            verdict = alert.vlm_person_verdicts.get(pr.person_idx)
            if pr.person_bbox:
                x1, y1, x2, y2 = (int(v) for v in pr.person_bbox)
                if verdict is False:
                    color = (180, 60, 180)  # viola: VLM ha scagionato
                    label = f"VLM: PPE presente? {', '.join(pr.missing_ppe)}"
                    any_refuted = True
                else:
                    color = (0, 45, 210)  # rosso: confermato o non validato
                    label = f"ALERT: {', '.join(pr.missing_ppe)}" + (" [VLM OK]" if verdict else "")
                cv.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                cv.putText(frame, label, (x1, max(y1 - 8, 14)), cv.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        if alert.vlm_confirmed is None:
            vlm_tag = "VLM pending"
        elif any_refuted:
            vlm_tag = "VLM: parzialmente scagionato" if alert.vlm_confirmed else "VLM: scagionato (falso positivo?)"
        elif alert.vlm_confirmed:
            vlm_tag = "VLM OK (confermato)"
        else:
            vlm_tag = "YOLO+Tracker (no VLM)"

        axes[i].imshow(cv.cvtColor(frame, cv.COLOR_BGR2RGB))
        axes[i].set_title(
            f"Alert #{i + 1} | t={alert.timestamp_s:.1f}s | {vlm_tag}\n{'; '.join(p.summary() for p in alert.violations)}",
            fontsize=8,
        )
        axes[i].axis("off")

    # Nascondi subplot vuoti
    for j in range(len(alerts), len(axes)):
        axes[j].set_visible(False)

    cap.release()
    plt.tight_layout()
    plt.show()

## 4 · Report JSON alert

In [ ]:
import json

if Path(ALERT_LOG).exists():
    data = json.loads(Path(ALERT_LOG).read_text())
    print(f"{len(data)} alert nel log\n")
    for entry in data:
        print(f"  frame={entry['frame']:>5}  t={entry['timestamp_s']:.1f}s  vlm_confirmed={entry['vlm_confirmed']}")
        for v in entry["violations"]:
            print(
                f"    Person#{v['person_idx']} mancanti={v['missing_ppe']} trovati={v['found_ppe']} vlm={v.get('vlm_confirmed')}"
            )
        if entry.get("vlm_response"):
            print(f"    VLM: {entry['vlm_response']}")
else:
    print("Log non ancora generato - esegui prima la cella di tracking.")